# Unsupervised Patient Phenotyping & Anomaly Detection
Trains `KNNImputer`, `StandardScaler`, `PCA`, `KMeans`, and `IsolationForest` on the real Diabetes 130-US Hospitals inpatient dataset (`Bena345/diabetes-readmission`) and saves `healthcare_unsupervised_phenotyper.pkl`.

In [ ]:
# !pip install -q datasets huggingface_hub scikit-learn joblib pandas numpy matplotlib


[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [1]:
from __future__ import annotations
from typing import Any, Dict, List, Tuple
import joblib
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest

/Users/yash/Library/A-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Real Inpatient Hospital Dataset (`Bena345/diabetes-readmission`)

In [2]:
csv_path = hf_hub_download(
    repo_id="Bena345/diabetes-readmission",
    filename="train.csv",
    repo_type="dataset"
)
diabetes_readmission_df = pd.read_csv(csv_path, low_memory=False)
diabetes_readmission_df.head()

,encounter_id,patient_nbr,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,157886028,82664136,5,74,3,18,0,0,0,9,...,No,Steady,No,No,No,No,No,Ch,Yes,No
1,102238272,84093948,9,23,6,22,0,0,0,9,...,No,No,No,No,No,No,No,No,No,No
2,104838786,69401565,1,25,1,1,1,0,0,2,...,No,No,No,No,No,No,No,No,No,No
3,65127540,10396926,12,54,2,15,0,0,0,9,...,No,Steady,No,No,No,No,No,No,Yes,No
4,410231108,64618560,2,38,0,7,0,0,0,9,...,No,No,No,No,No,No,No,No,No,No


In [3]:
from pandas.api.types import is_numeric_dtype

def load_tabular_readmission_dataset(csv_file_path: str, max_rows: int = 20000) -> Tuple[pd.DataFrame, np.ndarray, List[str]]:
    df_raw = pd.read_csv(csv_file_path, low_memory=False)
    print(f"Raw dataset shape: {df_raw.shape}")
    if len(df_raw) > max_rows:
        df_raw = df_raw.sample(n=max_rows, random_state=42).reset_index(drop=True)

    candidate_numeric = [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    ]
    available_cols = [c for c in candidate_numeric if c in df_raw.columns]

    if "age" in df_raw.columns and not is_numeric_dtype(df_raw["age"]):
        df_raw["age_years"] = (
            df_raw["age"]
            .astype(str)
            .str.extract(r"(\d+)")[0]
            .astype(float)
            + 5.0
        )
        available_cols.insert(0, "age_years")
    elif "age" in df_raw.columns:
        available_cols.insert(0, "age")

    X_df = df_raw[available_cols].apply(pd.to_numeric, errors="coerce")

    target_col = "readmitted" if "readmitted" in df_raw.columns else df_raw.columns[-1]
    raw_target = df_raw[target_col].astype(str).str.strip().str.upper()
    if set(raw_target.unique()).issubset({"0", "1", "0.0", "1.0"}):
        y = raw_target.astype(float).astype(int).to_numpy()
    else:
        y = raw_target.isin(["<30", "1", "TRUE", "YES"]).astype(int).to_numpy()

    print(f"Loaded {len(X_df):,} real inpatient encounters | Features: {available_cols} | Positive rate: {y.mean()*100:.2f}%")
    return X_df, y, available_cols

X_df, y_tab, feat_names = load_tabular_readmission_dataset(csv_path, max_rows=20000)
X_df.head()

Raw dataset shape: (96677, 50)
Loaded 20,000 real inpatient encounters | Features: ['age_years', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses'] | Positive rate: 11.34%


,age_years,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
0,85.0,4,63,0,5,0,0,1,9
1,75.0,2,1,3,11,0,0,0,4
2,65.0,9,42,3,19,0,0,0,9
3,75.0,4,60,0,11,1,0,6,8
4,65.0,5,26,3,11,0,0,0,9


In [4]:
type(pd.read_csv(csv_path)["age"].dtype)

pandas.StringDtype

## Train Unsupervised Phenotyper (`KNNImputer` + `StandardScaler` + `PCA` + `KMeans` + `IsolationForest`)

In [5]:
class SklearnUnsupervisedPhenotyper:
    """Production Unsupervised ML pipeline using scikit-learn."""

    def __init__(self, n_pca_components: int = 5, n_clusters: int = 3, contamination: float = 0.03):
        self.imputer = KNNImputer(n_neighbors=5)
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=n_pca_components, random_state=42)
        self.kmeans = KMeans(n_clusters=n_clusters, init="k-means++", n_init=10, random_state=42)
        self.iforest = IsolationForest(n_estimators=150, contamination=contamination, random_state=42, n_jobs=-1)
        self.feature_names_: List[str] = []
        self.cluster_labels_map_: Dict[int, str] = {}

    def fit(self, X_df: pd.DataFrame) -> "SklearnUnsupervisedPhenotyper":
        self.feature_names_ = list(X_df.columns)
        X_imp = self.imputer.fit_transform(X_df)
        X_scaled = self.scaler.fit_transform(X_imp)
        X_pca = self.pca.fit_transform(X_scaled)
        clusters = self.kmeans.fit_predict(X_pca)
        self.iforest.fit(X_scaled)

        acuity_scores = [X_scaled[clusters == c].mean() for c in range(self.kmeans.n_clusters)]
        order = np.argsort(acuity_scores)
        self.cluster_labels_map_ = {
            int(order[0]): "Phenotype 1: Low-Utilization Baseline Cohort",
            int(order[1]): "Phenotype 2: Moderate Chronic Multi-Medication Cohort",
            int(order[2]): "Phenotype 3: High-Acuity Frequent-Inpatient Cohort",
        }
        evr = np.sum(self.pca.explained_variance_ratio_) * 100
        print(f"PCA ({self.pca.n_components_} components) Explained Variance: {evr:.2f}%")
        for cid, label in self.cluster_labels_map_.items():
            print(f"  - Cluster {cid} ({label}): {int(np.sum(clusters == cid)):,} patients")
        return self

    def analyze_patient(self, patient_row: pd.DataFrame) -> Tuple[np.ndarray, Dict[str, Any]]:
        X_imp = self.imputer.transform(patient_row[self.feature_names_])
        X_scaled = self.scaler.transform(X_imp)
        X_pca = self.pca.transform(X_scaled)
        cid = int(self.kmeans.predict(X_pca)[0])
        anom_pred = int(self.iforest.predict(X_scaled)[0])
        anom_score = float(-self.iforest.score_samples(X_scaled)[0])

        return X_imp, {
            "cluster_id": cid,
            "phenotype_name": self.cluster_labels_map_.get(cid, f"Cluster {cid}"),
            "pca_coordinates": [round(float(x), 3) for x in X_pca[0]],
            "isolation_forest_anomaly_score": round(anom_score, 4),
            "is_outlier_flagged": bool(anom_pred == -1),
        }

In [6]:
unsup_engine = SklearnUnsupervisedPhenotyper(n_pca_components=5, n_clusters=3).fit(X_df)

PCA (5 components) Explained Variance: 72.42%
  - Cluster 0 (Phenotype 1: Low-Utilization Baseline Cohort): 12,348 patients
  - Cluster 2 (Phenotype 2: Moderate Chronic Multi-Medication Cohort): 6,273 patients
  - Cluster 1 (Phenotype 3: High-Acuity Frequent-Inpatient Cohort): 1,379 patients


## Save Trained Unsupervised Phenotyper (`healthcare_unsupervised_phenotyper.pkl`)

In [7]:
joblib.dump(unsup_engine, "healthcare_unsupervised_phenotyper.pkl")
print("Unsupervised phenotyper saved successfully to healthcare_unsupervised_phenotyper.pkl!")

Unsupervised phenotyper saved successfully to healthcare_unsupervised_phenotyper.pkl!
